In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
import joblib
import json

print('Libraries imported successfully')
# Load data from CSV
csv_path = 'cloudscheduling_cleaned.csv'
print(f"Loading data from: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dataset loaded: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

# Prepare features and target
target_col = 'Execution_Time (s)'
columns_to_drop = ['Task_ID', target_col]

# Remove columns that don't exist
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

X = df.drop(columns=columns_to_drop)
y = df[target_col]

print(f"\nFeatures: {list(X.columns)}")
print(f"Target: {target_col}")
print(f"\nTarget statistics:")
print(f"  Mean: {y.mean():.2f}")
print(f"  Std: {y.std():.2f}")
print(f"  Range: {y.min():.2f} - {y.max():.2f}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {len(X_train)}, Test samples: {len(X_test)}")

Libraries imported successfully
Loading data from: cloudscheduling_cleaned.csv
Dataset loaded: (20000, 9)

Columns: ['Task_ID', 'CPU_Usage (%)', 'RAM_Usage (MB)', 'Disk_IO (MB/s)', 'Network_IO (MB/s)', 'Priority', 'VM_ID', 'Execution_Time (s)', 'Target (Optimal Scheduling)']

First few rows:
   Task_ID  CPU_Usage (%)  RAM_Usage (MB)  Disk_IO (MB/s)  Network_IO (MB/s)  \
0  0.00000       0.078125        0.274211        0.648936           0.270833   
1  0.00005       0.421875        0.635121        0.840426           0.583333   
2  0.00010       0.500000        0.508349        0.340426           0.354167   
3  0.00015       0.453125        0.194254        0.872340           0.645833   
4  0.00020       0.734375        0.582257        0.393617           0.708333   

   Priority     VM_ID  Execution_Time (s)  Target (Optimal Scheduling)  
0       0.0  0.333333            0.623333                          0.0  
1       1.0  0.888889            0.456667                          0.0  
2      

# Neural Network Training Notebook
This notebook trains a neural network for energy consumption prediction using the cloudscheduling_cleaned.csv dataset and displays comprehensive accuracy metrics including R², RMSE, MAE, and Accuracy %.

In [2]:
# Train Neural Network
print("Training Neural Network model...")

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build neural network architecture
model = keras.Sequential([
    keras.layers.Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.1),
    
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='linear')
])

# Compile model
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Train with callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6)
]

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    verbose=1,
    callbacks=callbacks
)

print("\nTraining completed!")

Training Neural Network model...


c:\Users\91967\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2170 - mae: 0.3436 - val_loss: 0.0520 - val_mae: 0.1829 - learning_rate: 0.0010
Epoch 2/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0433 - mae: 0.1653 - val_loss: 0.0258 - val_mae: 0.1335 - learning_rate: 0.0010
Epoch 3/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0303 - mae: 0.1425 - val_loss: 0.0229 - val_mae: 0.1278 - learning_rate: 0.0010
Epoch 4/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0269 - mae: 0.1358 - val_loss: 0.0231 - val_mae: 0.1282 - learning_rate: 0.0010
Epoch 5/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0251 - mae: 0.1324 - val_loss: 0.0216 - val_mae: 0.1251 - learning_rate: 0.0010
Epoch 6/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0242 - mae: 0.1310 - val_loss: 0.0224 - val_mae: 0.1265 - learning_rate: 0.0010
Epoch 7/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0243 - mae: 0.1317 - val_loss: 0.0222 - val_mae: 0.1262 - learning_rate: 0.0010

In [6]:
# Evaluate model
y_pred = model.predict(X_test_scaled, verbose=0).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Calculate accuracy as percentage (based on mean absolute percentage error)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
accuracy = 100 - mape

print('\n' + '='*50)
print('NEURAL NETWORK EVALUATION RESULTS')
print('='*50)
print(f'R² Score: {r2:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'Accuracy: {accuracy:.2f}%')
print('='*50)

results = {
    'r2': float(r2),
    'rmse': float(rmse),
    'mae': float(mae),
    'mse': float(mse),
    'accuracy_percent': float(accuracy),
    'training_epochs': len(history.history['loss'])
}


NEURAL NETWORK EVALUATION RESULTS
R² Score: 0.7413
RMSE: 0.1470
MAE: 0.1264
MSE: 0.0216
Accuracy: -inf%
